# Scenario: Async OpenAI Client Coverage

Validates that `AsyncOpenAI` from `openai` operates properly with async/await methods against the OGX server:
- **Server Health:** `client.get("/health")`
- **Models:** `client.models.list()`
- **Responses API:** `client.responses.create()`
- **Chat Completions:** `client.chat.completions.create()`
- **Embeddings:** `client.embeddings.create()`

## Setup & Initialization

Load configuration from environment variables and initialize `AsyncOpenAI`.

In [ ]:
import os
from openai import AsyncOpenAI
from scripts.helpers import response_text

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("INFERENCE_MODEL")
embedding_model = os.environ.get("EMBEDDING_MODEL")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "768"))

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

openai_base_url = base_url.rstrip("/")
openai_base_url = (
    openai_base_url if openai_base_url.endswith("/v1") else openai_base_url + "/v1"
)

client = AsyncOpenAI(api_key="no-key-needed", base_url=openai_base_url)

## Server Health Check (`/v1/health`)

Verify `await client.get("/health")` completes and returns status OK.

In [ ]:
health_resp = await client.get("/health", cast_to=object)
assert health_resp is not None, "Expected health response"
health_status = (
    health_resp.get("status")
    if isinstance(health_resp, dict)
    else getattr(health_resp, "status", str(health_resp))
)
assert health_status == "OK" or health_resp == "OK", (
    f"Health check failed or unexpected response: {health_resp!r}"
)

## Models List (`models.list`)

Verify `await client.models.list()` returns the list of registered models.

In [ ]:
models_resp = await client.models.list()
assert models_resp is not None, "Expected models response"
model_ids = [m.id for m in models_resp.data]

assert len(model_ids) > 0, "Expected at least one model in list"
assert any(model in mid or mid in model for mid in model_ids), (
    f"Configured model {model!r} not found in model IDs: {model_ids}"
)

## Responses API (`responses.create`)

Verify async responses creation with `await client.responses.create()`.

In [ ]:
response = await client.responses.create(
    model=model,
    input="Explain disestablishmentarianism to a smart five year old.",
)
assert response is not None, "Expected response object"
assert getattr(response, "status", "completed") == "completed"
out_text = getattr(response, "output_text", None) or response_text(response)
assert out_text and len(out_text.strip()) > 0, (
    "Expected non-empty output text from responses.create"
)

## Chat Completions (`chat.completions.create`)

Verify async chat completion with `await client.chat.completions.create()`.

In [ ]:
chat_resp = await client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Reply with exactly one word: Hello"}],
    temperature=0.0,
)
assert chat_resp is not None, "Expected chat completion response"
assert hasattr(chat_resp, "choices") and len(chat_resp.choices) > 0, (
    "Expected non-empty choices"
)
content = chat_resp.choices[0].message.content
assert content and len(content.strip()) > 0, "Expected non-empty message content"

## Embeddings (`embeddings.create`)

Verify async embedding creation with `await client.embeddings.create()` if an embedding model is configured.

In [ ]:
if embedding_model:
    emb_resp = await client.embeddings.create(
        model=embedding_model,
        input="Async client embedding test",
    )
    assert emb_resp is not None, "Expected embedding response"
    assert hasattr(emb_resp, "data") and len(emb_resp.data) > 0, (
        "Expected embedding data"
    )
    vector = emb_resp.data[0].embedding
    assert len(vector) == embedding_dimension, (
        f"Expected vector dimension {embedding_dimension}, got {len(vector)}"
    )
else:
    print("EMBEDDING_MODEL not set, skipping async embedding test")
    assert True

## Context Manager & Cleanup

Verify that `AsyncOpenAI` works as an async context manager and closes properly.

In [ ]:
async with AsyncOpenAI(
    api_key="no-key-needed", base_url=openai_base_url
) as async_client:
    models = await async_client.models.list()
    assert models is not None

await client.close()